In [ ]:
import rdkit
import pandas as pd

In [ ]:
df = pd.read_csv("../data/PIK3CA_deduplicated.csv") # Deduplicated curated set; descriptors and pIC50 are added below
display(df_2classes)

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

Compute molecular weight (MW), octanol/water partition coefficient (LogP), and
hydrogen-bond donor/acceptor counts for each molecule from its SMILES string

In [ ]:
def lipinski(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem)
        moldata.append(mol)

    baseData= np.arange(1,1)
    i=0
    for mol in moldata:

        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)

        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])

        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1

    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)

    return descriptors

In [ ]:
df_lipinski = lipinski(df.canonical_smiles)
df_lipinski

In [ ]:
df

In [ ]:
df_combinado = pd.concat([df,df_lipinski], axis = 1)
display(df_combinado)

In [ ]:
df_combinado.standard_value.describe()

Convert IC50 (nM) values to pIC50

In [ ]:
import numpy as np

def pIC50(input):
    pIC50 = []

    for i in input['standard_value_norm']:
        molar = i*(10**-9) # Converts nM to M
        pIC50.append(-np.log10(molar))

    input['pIC50'] = pIC50
    x = input.drop('standard_value_norm', axis=1)

    return x

In [ ]:
-np.log10( (10**-9)* 100000000 )

In [ ]:
-np.log10( (10**-9)* 10000000000 )

Cap extreme IC50 values at 100 uM before the log transform

In [ ]:
def norm_value(input):
    norm = []

    for i in input['standard_value']:
        if i > 100000000:
          i = 100000000
        norm.append(i)

    input['standard_value_norm'] = norm
    x = input.drop('standard_value', axis=1)

    return x

In [ ]:
df_normal = norm_value(df_combinado)
display(df_normal)

In [ ]:
df_normal.describe()

In [ ]:
df_final = pIC50(df_normal)
display(df_final)

In [ ]:
df2 = df_final["pIC50"]
df3 = df_final["canonical_smiles"]
df4 = pd.concat([df3, df2], axis = 1)
df4.to_csv('PIK3CA_pic50_full.csv')

In [ ]:
df_final.to_csv('PIK3CA_classes.csv')

In [ ]:
df_final.pIC50.describe()

In [ ]:
df_2classes = df_final[df_final.classe_bioatividade != 'intermediary']
df_2classes

Exploratory plots

In [ ]:
import seaborn as sns
sns.set(style='ticks')
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
# Default font size
plt.rcParams['font.size'] = 12

# Plot
plt.figure(figsize=(5.5, 5.5))
sns.countplot(x='classe_bioatividade', data=df_2classes, edgecolor='black', palette='Set2')

# Axis labels
plt.xlabel('Bioactivity Class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')


# plt.xticks(rotation=45)

# Margins so the y-label is fully visible
plt.subplots_adjust(left=0.18, bottom=0.15, right=0.95, top=0.95)





# Save the figure
plt.savefig('bioactivity_distribution.png', dpi=300, bbox_inches='tight')



#plt.savefig('PIK3CA_plot_bioactivity_class.png')

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Scatter plot sized to fit the external legend
plt.figure(figsize=(7, 6))

# Light colour palette for the three bioactivity classes
custom_palette = {
    'active':       '#99FF99',  
    'intermediate': '#FF9999',  
    'inactive':     '#99CCFF'   
}

# Scatter plot

sns.scatterplot(
    x='MW',           
    y='LogP',
    data=df_2classes,               
    hue='classe_bioatividade',      
    palette='Set2',         
    size='pIC50',                   
    sizes=(20, 120),                
    edgecolor='black',
    linewidth=0.5,                  
    alpha=0.7
)

# Axis labels
plt.xlabel('Molecular Weight (g/mol)', fontsize=14, fontweight='bold')
plt.ylabel('LogP (oil/water)', fontsize=14, fontweight='bold')

# Move the legend outside the axes
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0,
    title='Bioactivity Class\n&pIC50',
    fontsize=12,
    title_fontsize=13
)

# Margins: keep labels and legend inside the figure
plt.subplots_adjust(left=0.15, bottom=0.12, right=0.78, top=0.95)

# Save as PDF and PNG
plt.savefig('plot_MW_vs_LogP_3classes_light-1.pdf', dpi=300, bbox_inches='tight')
plt.savefig('plot_MW_vs_LogP_3classes_light-1.png', dpi=300, bbox_inches='tight')

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Box plot layout
plt.figure(figsize=(8, 8))

# Matching light colour palette
custom_palette = {
    'active':       '#99FF99',  
    'intermediate': '#FF9999',  
    'inactive':     '#99CCFF'   
}

# Box plot
sns.boxplot(
    x='classe_bioatividade',
    y='pIC50',
    data=df_2classes,
    palette='Set2',
    width=0.6,
    linewidth=2
)

# Optionally overlay a swarm plot to show individual points

sns.swarmplot(
    x='classe_bioatividade',
    y='pIC50',
    data=df_2classes,
    color='black',
    alpha=0.4,
    size=4
)

# Axis labels
plt.xlabel('Bioactivity Class', fontsize=14, fontweight='bold')
plt.ylabel('pIC50 Value', fontsize=14, fontweight='bold')





# Margins so labels are not clipped
plt.subplots_adjust(left=0.18, bottom=0.15, right=0.95, top=0.95)

plt.savefig('boxplot_pIC50_by_bioactivity.pdf', dpi=300, bbox_inches='tight')
plt.savefig('boxplot_pIC50_by_bioactivity.png', dpi=300, bbox_inches='tight')


